# Cluster Forecasting Experiment Runner

This notebook keeps the forecasting features built from raw 2023 daily usage data.
Cluster labels are only used to route households into cluster-specific models.

In [ ]:
from pathlib import Path
import os
import sys
import subprocess

REPO_NAME = "Clustering-And-Forecasting-TimeSeries-PlayingGround"
REPO_URL = "https://github.com/QuirkyCroissant/Clustering-And-Forecasting-TimeSeries-PlayingGround.git"
BRANCH = os.environ.get("NOTEBOOK_GIT_BRANCH", "feat/cluster-forecasting-lgbm-xgb")

kaggle_root = Path("/kaggle/working")
cwd = Path.cwd().resolve()

if cwd.name == REPO_NAME and (cwd / "src" / "forecasting").exists():
    REPO_ROOT = cwd
elif kaggle_root.exists():
    repo_dir = kaggle_root / REPO_NAME
    if not repo_dir.exists():
        subprocess.run(
            ["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(repo_dir)],
            check=True,
        )
    os.chdir(repo_dir)
    REPO_ROOT = repo_dir.resolve()
else:
    raise RuntimeError(
        "Could not determine REPO_ROOT automatically. Open the notebook from the repo root or run it on Kaggle."
    )

SRC_PATH = REPO_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

print("REPO_ROOT:", REPO_ROOT)
print("SRC_PATH:", SRC_PATH)

In [ ]:
import json
from datetime import datetime, timezone

import pandas as pd
from IPython.display import display

from forecasting.data import (
    discover_cluster_cases,
    ensure_experiment_dirs,
    ensure_output_dirs,
    load_cluster_labels,
    load_static_features,
    load_wide_csv,
)
from forecasting.evaluate import (
    evaluate_global_and_cluster,
    plot_sample_households_by_group,
)
from forecasting.features import make_training_frame
from forecasting.predict import forecast_by_group, forecast_global
from forecasting.train import fit_cluster_models, fit_global_model

OUT = ensure_output_dirs(REPO_ROOT)

TRAIN_23_PATH = REPO_ROOT / "data" / "raw" / "sample_23.csv"
TEST_24_PATH = REPO_ROOT / "data" / "raw" / "sample_24.csv"
CLUSTER_CASE_DIR = REPO_ROOT / "notebooks" / "outputs" / "feature"
SHAPELET_STATIC_PATH = REPO_ROOT / "notebooks" / "outputs" / "shapelet_experiments" / "shapelet_features.csv"

In [ ]:
DEBUG = False
DEBUG_FRAC = 0.2

MODEL_NAME = "xgb"
GPU_ENABLED = True
MIN_CLUSTER_ROWS = 500
MIN_CLUSTER_HOUSEHOLDS = 30

SELECTED_CASES = None
INCLUDE_BASE_VARIANT = True
INCLUDE_SHAPELET_STATIC_VARIANT = True

PLOT_SAMPLE_PER_GROUP = 2
PLOT_MAX_GROUPS = None
RANDOM_STATE = 42

In [ ]:
def build_experiment_configs(case_paths: dict[str, Path]) -> list[dict]:
    case_names = SELECTED_CASES or list(case_paths)
    missing = sorted(set(case_names) - set(case_paths))
    if missing:
        raise ValueError(f"Unknown case names requested: {missing}")

    variant_defs = []
    if INCLUDE_BASE_VARIANT:
        variant_defs.append(("base", None))
    if INCLUDE_SHAPELET_STATIC_VARIANT:
        if not SHAPELET_STATIC_PATH.exists():
            raise FileNotFoundError(f"Missing shapelet static feature file: {SHAPELET_STATIC_PATH}")
        variant_defs.append(("shapelet_static", SHAPELET_STATIC_PATH))

    configs = []
    for case_name in case_names:
        for variant_name, static_path in variant_defs:
            configs.append(
                {
                    "case_name": case_name,
                    "variant_name": variant_name,
                    "experiment_name": f"{case_name}_{variant_name}",
                    "cluster_path": case_paths[case_name],
                    "static_features_path": static_path,
                }
            )
    return configs


def maybe_apply_debug_subset(train_23, test_24, cluster_labels, static_features=None):
    if not DEBUG:
        return train_23, test_24, cluster_labels, static_features

    sampled_ids = train_23["ID"].sample(frac=DEBUG_FRAC, random_state=RANDOM_STATE)
    sampled_ids = set(sampled_ids.astype(str))

    train_23 = train_23[train_23["ID"].astype(str).isin(sampled_ids)].reset_index(drop=True)
    test_24 = test_24[test_24["ID"].astype(str).isin(sampled_ids)].reset_index(drop=True)
    cluster_labels = cluster_labels[cluster_labels["ID"].astype(str).isin(sampled_ids)].reset_index(drop=True)

    if static_features is not None:
        static_features = static_features[static_features["ID"].astype(str).isin(sampled_ids)].reset_index(drop=True)

    return train_23, test_24, cluster_labels, static_features


def write_json(path: Path, payload):
    path.write_text(json.dumps(payload, indent=2, default=str), encoding="utf-8")


def run_experiment(exp_config, train_23_base, test_24_base):
    import matplotlib.pyplot as plt

    exp_dirs = ensure_experiment_dirs(REPO_ROOT, exp_config["experiment_name"])
    cluster_labels = load_cluster_labels(exp_config["cluster_path"])
    static_features = None
    if exp_config["static_features_path"] is not None:
        static_features = load_static_features(exp_config["static_features_path"])

    train_23, test_24, cluster_labels, static_features = maybe_apply_debug_subset(
        train_23=train_23_base.copy(),
        test_24=test_24_base.copy(),
        cluster_labels=cluster_labels,
        static_features=static_features,
    )

    train_df = make_training_frame(
        train_wide=train_23,
        cluster_labels=cluster_labels,
        static_features=static_features,
        show_progress=True,
    )
    feature_cols = [
        c for c in train_df.columns
        if c not in ["ID", "ds", "ForecastGroup", "target"]
    ]
    future_dates = pd.to_datetime(test_24.columns[1:])

    global_fit = fit_global_model(
        train_df=train_df,
        feature_cols=feature_cols,
        model_name=MODEL_NAME,
        use_gpu=GPU_ENABLED,
        verbose=50,
    )
    cluster_fits = fit_cluster_models(
        train_df=train_df,
        feature_cols=feature_cols,
        model_name=MODEL_NAME,
        min_households=MIN_CLUSTER_HOUSEHOLDS,
        min_rows=MIN_CLUSTER_ROWS,
        use_gpu=GPU_ENABLED,
        show_progress=True,
        verbose=False,
    )

    pred_global = forecast_global(
        train_23_wide=train_23,
        future_dates=future_dates,
        model=global_fit["model"],
        static_features=static_features,
        show_progress=True,
        feature_cols=feature_cols,
    )
    pred_cluster = forecast_by_group(
        train_23_wide=train_23,
        cluster_labels=cluster_labels,
        future_dates=future_dates,
        group_models=cluster_fits,
        fallback_model=global_fit["model"],
        static_features=static_features,
        show_progress=True,
        feature_cols=feature_cols,
    )

    eval_tables = evaluate_global_and_cluster(
        pred_global_wide=pred_global,
        pred_cluster_wide=pred_cluster,
        truth_wide=test_24,
        cluster_labels=cluster_labels,
        trained_groups=cluster_fits.keys(),
        global_model_label=f"global_{MODEL_NAME}",
        cluster_model_label=f"cluster_{MODEL_NAME}",
    )

    sampled_households, fig = plot_sample_households_by_group(
        train_23_wide=train_23,
        test_24_wide=test_24,
        pred_cluster_wide=pred_cluster,
        cluster_labels=cluster_labels,
        mae_cluster_df=eval_tables["mae_cluster_detail"],
        pred_global_wide=pred_global,
        bucket_col="SparsityBucket",
        cluster_col="ForecastGroup",
        n_per_group=PLOT_SAMPLE_PER_GROUP,
        random_state=RANDOM_STATE,
        max_groups=PLOT_MAX_GROUPS,
    )

    pred_global.to_csv(exp_dirs["predictions"] / f"pred_2024_global_{MODEL_NAME}.csv", index=False)
    pred_cluster.to_csv(exp_dirs["predictions"] / f"pred_2024_cluster_{MODEL_NAME}.csv", index=False)
    sampled_households.to_csv(exp_dirs["plots"] / "sampled_households.csv", index=False)

    fig.savefig(exp_dirs["plots"] / "sample_households_by_group.png", dpi=200, bbox_inches="tight")
    plt.close(fig)

    metric_frames = {
        "overall_summary": eval_tables["overall_summary"],
        "mae_global_detail": eval_tables["mae_global_detail"],
        "mae_cluster_detail": eval_tables["mae_cluster_detail"],
        "route_summary": eval_tables["route_summary"],
        "cluster_mae_summary": eval_tables["cluster_mae_summary"],
        "compare_detail": eval_tables["compare_detail"],
        "cluster_compare_summary": eval_tables["cluster_compare_summary"],
    }
    for table_name, df in metric_frames.items():
        df.assign(experiment_name=exp_config["experiment_name"]).to_csv(
            exp_dirs["metrics"] / f"{table_name}.csv",
            index=False,
        )

    experiment_metadata = {
        "experiment_name": exp_config["experiment_name"],
        "case_name": exp_config["case_name"],
        "variant_name": exp_config["variant_name"],
        "cluster_path": str(exp_config["cluster_path"]),
        "static_features_path": str(exp_config["static_features_path"]) if exp_config["static_features_path"] else None,
        "model_name": MODEL_NAME,
        "gpu_enabled": GPU_ENABLED,
        "debug": DEBUG,
        "debug_frac": DEBUG_FRAC,
        "random_state": RANDOM_STATE,
        "train_shape": list(train_23.shape),
        "test_shape": list(test_24.shape),
        "train_frame_shape": list(train_df.shape),
        "feature_cols": feature_cols,
        "n_feature_cols": len(feature_cols),
        "trained_groups": sorted(map(str, cluster_fits.keys())),
        "n_trained_cluster_models": len(cluster_fits),
        "global_fit_metadata": global_fit["metadata"],
        "cluster_fit_metadata": {
            str(group): fit["metadata"] for group, fit in cluster_fits.items()
        },
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
    }

    write_json(exp_dirs["metadata"] / "experiment_metadata.json", experiment_metadata)
    write_json(exp_dirs["metadata"] / "global_eval_log.json", global_fit["eval_log"])
    write_json(
        exp_dirs["metadata"] / "cluster_eval_logs.json",
        {str(group): fit["eval_log"] for group, fit in cluster_fits.items()},
    )

    return {
        "experiment_name": exp_config["experiment_name"],
        "config": exp_config,
        "dirs": exp_dirs,
        "overall_summary": eval_tables["overall_summary"].assign(experiment_name=exp_config["experiment_name"]),
        "route_summary": eval_tables["route_summary"].assign(experiment_name=exp_config["experiment_name"]),
        "cluster_mae_summary": eval_tables["cluster_mae_summary"].assign(experiment_name=exp_config["experiment_name"]),
        "cluster_compare_summary": eval_tables["cluster_compare_summary"].assign(experiment_name=exp_config["experiment_name"]),
    }

In [ ]:
train_23 = load_wide_csv(TRAIN_23_PATH)
test_24 = load_wide_csv(TEST_24_PATH)

cluster_case_paths = discover_cluster_cases(CLUSTER_CASE_DIR)
experiment_configs = build_experiment_configs(cluster_case_paths)

experiment_plan = pd.DataFrame(experiment_configs)
display(experiment_plan)
print(f"Planned experiments: {len(experiment_configs)}")

In [ ]:
experiment_runs = []

for exp_config in experiment_configs:
    print(f"\n=== Running {exp_config['experiment_name']} ===")
    experiment_runs.append(run_experiment(exp_config, train_23, test_24))

In [ ]:
all_overall_summary = pd.concat(
    [run["overall_summary"] for run in experiment_runs],
    ignore_index=True,
)
all_route_summary = pd.concat(
    [run["route_summary"] for run in experiment_runs],
    ignore_index=True,
)
all_cluster_mae_summary = pd.concat(
    [run["cluster_mae_summary"] for run in experiment_runs],
    ignore_index=True,
)
all_cluster_compare_summary = pd.concat(
    [run["cluster_compare_summary"] for run in experiment_runs],
    ignore_index=True,
)

model_comparison = (
    all_overall_summary
    .pivot(index="experiment_name", columns="model", values="mean_mae")
    .reset_index()
)
model_comparison["delta_cluster_minus_global"] = (
    model_comparison[f"cluster_{MODEL_NAME}"] - model_comparison[f"global_{MODEL_NAME}"]
)
model_comparison = model_comparison.sort_values("delta_cluster_minus_global").reset_index(drop=True)

all_overall_summary.to_csv(OUT["metrics"] / "all_experiments_overall_summary.csv", index=False)
all_route_summary.to_csv(OUT["metrics"] / "all_experiments_route_summary.csv", index=False)
all_cluster_mae_summary.to_csv(OUT["metrics"] / "all_experiments_cluster_mae_summary.csv", index=False)
all_cluster_compare_summary.to_csv(OUT["metrics"] / "all_experiments_cluster_compare_summary.csv", index=False)
model_comparison.to_csv(OUT["metrics"] / "all_experiments_model_comparison.csv", index=False)

display(all_overall_summary.sort_values(["experiment_name", "model"]).reset_index(drop=True))
display(model_comparison)
display(all_cluster_compare_summary.head(20))